# Module 2: Building & Registering MCP Servers

## Build a GitHub MCP Server with the Python MCP SDK

In this module you'll learn how to:
- Create an MCP server using the **Python MCP SDK** (`FastMCP`)
- Register tools that wrap a real API (GitHub REST API)
- Understand how `list_tools` and `call_tool` work under the hood
- Run the server locally and connect it to Cursor

We'll build a server with **three tools**:

| Tool | What it does |
|------|-------------|
| `get_repos` | List repositories for a GitHub user/org |
| `get_pull_requests` | List pull requests for a repository |
| `write_pull_request_review` | Post a review comment on a PR |

```
AI Assistant  ──MCP──►  Our Server  ──HTTP──►  GitHub API
```

---

## Step 1: Install Dependencies

In [ ]:
!pip install mcp httpx python-dotenv

## Step 2: Setup — Load token & create the MCP server

We use `FastMCP` from the MCP SDK.  
A `GITHUB_TOKEN` is optional for read-only tools but **required** for `write_pull_request_review`.

Make sure `module-02/.env` contains:
```
GITHUB_TOKEN=ghp_your_token_here
```

In [ ]:
import os
import httpx
from dotenv import load_dotenv
from mcp.server.fastmcp import FastMCP

load_dotenv("../.env")

GITHUB_API_BASE = "https://api.github.com"
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

print(f"Token loaded: {'Yes' if GITHUB_TOKEN else 'No'}")

# Create the MCP server instance
mcp = FastMCP("github-tools")
print(f"MCP server created: {mcp.name}")

### Helper: GitHub API client

A thin wrapper so every tool can call the GitHub REST API without repeating boilerplate.

In [ ]:
def _headers() -> dict:
    h = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }
    if GITHUB_TOKEN:
        h["Authorization"] = f"Bearer {GITHUB_TOKEN}"
    return h


async def _github(method: str, endpoint: str, **kwargs) -> dict | list:
    """Fire a request against the GitHub REST API and return parsed JSON."""
    async with httpx.AsyncClient() as client:
        resp = await client.request(
            method,
            f"{GITHUB_API_BASE}{endpoint}",
            headers=_headers(),
            **kwargs,
        )
        if resp.status_code >= 400:
            return {"error": resp.status_code, "message": resp.text}
        return resp.json()

print("GitHub helper ready.")

---

## Step 3: Define the Tools

Each tool is a regular `async` Python function decorated with `@mcp.tool()`.

The MCP SDK automatically:
1. **Generates a JSON Schema** from the function's type hints and docstring
2. **Registers** it so any MCP client can discover it via `list_tools`
3. **Routes** incoming `call_tool` requests to the right function

### Tool 1 — `get_repos`

Lists public repositories for a given GitHub user or organisation.

In [ ]:
@mcp.tool()
async def get_repos(
    owner: str,
    per_page: int = 10,
    page: int = 1,
) -> str:
    """List public repositories for a GitHub user or organisation."""
    result = await _github(
        "GET",
        f"/users/{owner}/repos",
        params={"per_page": per_page, "page": page, "sort": "updated"},
    )
    if isinstance(result, list):
        repos = [
            {
                "name": r["name"],
                "full_name": r["full_name"],
                "description": r.get("description", ""),
                "stars": r["stargazers_count"],
                "language": r.get("language"),
                "url": r["html_url"],
            }
            for r in result
        ]
        return str(repos)
    return str(result)

print("Tool registered: get_repos")

### Tool 2 — `get_pull_requests`

Lists pull requests for a given repository. Supports filtering by state (`open`, `closed`, `all`).

In [ ]:
@mcp.tool()
async def get_pull_requests(
    owner: str,
    repo: str,
    state: str = "open",
    per_page: int = 10,
    page: int = 1,
) -> str:
    """List pull requests for a repository. State can be open, closed, or all."""
    result = await _github(
        "GET",
        f"/repos/{owner}/{repo}/pulls",
        params={"state": state, "per_page": per_page, "page": page},
    )
    if isinstance(result, list):
        prs = [
            {
                "number": pr["number"],
                "title": pr["title"],
                "state": pr["state"],
                "user": pr["user"]["login"],
                "created_at": pr["created_at"],
                "url": pr["html_url"],
            }
            for pr in result
        ]
        return str(prs)
    return str(result)

print("Tool registered: get_pull_requests")

### Tool 3 — `write_pull_request_review`

Posts a review on a pull request. This is a **write** operation and requires a `GITHUB_TOKEN` with repo write access.

In [ ]:
@mcp.tool()
async def write_pull_request_review(
    owner: str,
    repo: str,
    pull_number: int,
    body: str,
    event: str = "COMMENT",
) -> str:
    """Post a review on a pull request.

    event must be one of COMMENT, APPROVE, or REQUEST_CHANGES.
    Requires a GITHUB_TOKEN with write access to the repository.
    """
    if not GITHUB_TOKEN:
        return str({"error": "GITHUB_TOKEN is required to write reviews."})
    if event not in ("COMMENT", "APPROVE", "REQUEST_CHANGES"):
        return str({"error": f"Invalid event '{event}'. Use COMMENT, APPROVE, or REQUEST_CHANGES."})

    result = await _github(
        "POST",
        f"/repos/{owner}/{repo}/pulls/{pull_number}/reviews",
        json={"event": event, "body": body},
    )
    return str(result)

print("Tool registered: write_pull_request_review")

---

## Step 4: Inspect — `list_tools`

MCP clients discover available tools by calling `list_tools`.  
Let's see what our server exposes — note how the SDK auto-generated the JSON Schema from our type hints.

In [ ]:
import json

tools = await mcp.list_tools()

print(f"Server exposes {len(tools)} tools:\n")
for tool in tools:
    print(f"  • {tool.name}")
    print(f"    {tool.description}")
    print(f"    Schema: {json.dumps(tool.inputSchema, indent=2)}")
    print()

---

## Step 5: Test — `call_tool`

MCP clients invoke tools via `call_tool(name, arguments)`.  
We can simulate exactly what a client does by calling `mcp.call_tool()` with a tool name and arguments dict.

`call_tool` returns a tuple of `(content_blocks, output_dict)` — the content blocks are what get sent back to the AI assistant.

### 5a. `get_repos` — list repos for a user

In [ ]:
content_blocks, output = await mcp.call_tool("get_repos", {"owner": "octocat", "per_page": 5})

print("call_tool('get_repos', {owner: 'octocat'}):\n")
for block in content_blocks:
    print(block.text)

### 5b. `get_pull_requests` — list PRs for a repo

In [ ]:
content_blocks, output = await mcp.call_tool("get_pull_requests", {"owner": "octocat", "repo": "hello-world", "state": "all", "per_page": 5})

print("call_tool('get_pull_requests', {owner: 'octocat', repo: 'hello-world'}):\n")
for block in content_blocks:
    print(block.text)

### 5c. `write_pull_request_review` — post a review (requires token + write access)

**Uncomment and update** the cell below to test against a repo you own.

In [ ]:
# Uncomment to test — replace with a repo and PR you have write access to:
#
# content_blocks, output = await mcp.call_tool("write_pull_request_review", {
#     "owner": "YOUR_USERNAME",
#     "repo": "YOUR_REPO",
#     "pull_number": 1,
#     "body": "Workshop test: MCP SDK review tool works!",
#     "event": "COMMENT",
# })
# for block in content_blocks:
#     print(block.text)

print("Skipped — uncomment and set owner/repo/pull_number to test.")

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 348, in dispatch_control
    await self.process_control(msg)
  File "/Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 354, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list


[04/21/26 16:03:14] ERROR    Uncaught exception in ZMQStream callback                              ]8;id=148962;file:///Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py\zmqstream.py]8;;\:]8;id=688117;file:///Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py#567\567]8;;\
                             ╭──────────────── Traceback (most recent call last) ────────────────╮                 
                             │ /Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/pyth │                 
                             │ on3.12/site-packages/zmq/eventloop/zmqstream.py:565 in _log_error │                 
                             │                                                                   │                 
                             │   562 │   │   │   # handle async callbacks                        │                 
                             │   563 │   │   │   def _log_error(f):                              │                 
                             │   564 │   │   │   │   try:                                        │                 
                             │ ❱ 565 │   │   │   │   │   f.result()                              │                 
                             │   566 │   │   │   │   except Exception:                           │                 
                             │   567 │   │   │   │   │   gen_log.error(                          │                 
                             │   568 │   │   │   │   │   │   "Uncaught exception in ZMQStream ca │                 
                             │                                                                   │                 
                             │ /Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/pyth │                 
                             │ on3.12/site-packages/ipykernel/kernelbase.py:348 in               │                 
                             │ dispatch_control                                                  │                 
                             │                                                                   │                 
                             │    345 │   │   """Dispatch a control request, ensuring only one m │                 
                             │        time."""                                                   │                 
                             │    346 │   │   # Ensure only one control message is processed at  │                 
                             │    347 │   │   async with self._control_lock:                     │                 
                             │ ❱  348 │   │   │   await self.process_control(msg)                │                 
                             │    349 │                                                          │                 
                             │    350 │   async def process_control(self, msg):                  │                 
                             │    351 │   │   """dispatch control requests"""                    │                 
                             │                                                                   │                 
                             │ /Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/pyth │                 
                             │ on3.12/site-packages/ipykernel/kernelbase.py:354 in               │                 
                             │ process_control                                                   │                 
                             │                                                                   │                 
                             │    351 │   │   """dispatch control requests"""                    │                 
                             │    352 │   │   if not self.session:                               │                 


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 348, in dispatch_control
    await self.process_control(msg)
  File "/Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 354, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list


[04/21/26 16:03:15] ERROR    Uncaught exception in ZMQStream callback                              ]8;id=911483;file:///Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py\zmqstream.py]8;;\:]8;id=511768;file:///Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py#567\567]8;;\
                             ╭──────────────── Traceback (most recent call last) ────────────────╮                 
                             │ /Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/pyth │                 
                             │ on3.12/site-packages/zmq/eventloop/zmqstream.py:565 in _log_error │                 
                             │                                                                   │                 
                             │   562 │   │   │   # handle async callbacks                        │                 
                             │   563 │   │   │   def _log_error(f):                              │                 
                             │   564 │   │   │   │   try:                                        │                 
                             │ ❱ 565 │   │   │   │   │   f.result()                              │                 
                             │   566 │   │   │   │   except Exception:                           │                 
                             │   567 │   │   │   │   │   gen_log.error(                          │                 
                             │   568 │   │   │   │   │   │   "Uncaught exception in ZMQStream ca │                 
                             │                                                                   │                 
                             │ /Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/pyth │                 
                             │ on3.12/site-packages/ipykernel/kernelbase.py:348 in               │                 
                             │ dispatch_control                                                  │                 
                             │                                                                   │                 
                             │    345 │   │   """Dispatch a control request, ensuring only one m │                 
                             │        time."""                                                   │                 
                             │    346 │   │   # Ensure only one control message is processed at  │                 
                             │    347 │   │   async with self._control_lock:                     │                 
                             │ ❱  348 │   │   │   await self.process_control(msg)                │                 
                             │    349 │                                                          │                 
                             │    350 │   async def process_control(self, msg):                  │                 
                             │    351 │   │   """dispatch control requests"""                    │                 
                             │                                                                   │                 
                             │ /Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/pyth │                 
                             │ on3.12/site-packages/ipykernel/kernelbase.py:354 in               │                 
                             │ process_control                                                   │                 
                             │                                                                   │                 
                             │    351 │   │   """dispatch control requests"""                    │                 
                             │    352 │   │   if not self.session:                               │                 


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 348, in dispatch_control
    await self.process_control(msg)
  File "/Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 354, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list


...

[04/21/26 16:04:33] ERROR    Uncaught exception in ZMQStream callback                              ]8;id=158884;file:///Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py\zmqstream.py]8;;\:]8;id=971770;file:///Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py#567\567]8;;\
                             ╭──────────────── Traceback (most recent call last) ────────────────╮                 
                             │ /Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/pyth │                 
                             │ on3.12/site-packages/zmq/eventloop/zmqstream.py:565 in _log_error │                 
                             │                                                                   │                 
                             │   562 │   │   │   # handle async callbacks                        │                 
                             │   563 │   │   │   def _log_error(f):                              │                 
                             │   564 │   │   │   │   try:                                        │                 
                             │ ❱ 565 │   │   │   │   │   f.result()                              │                 
                             │   566 │   │   │   │   except Exception:                           │                 
                             │   567 │   │   │   │   │   gen_log.error(                          │                 
                             │   568 │   │   │   │   │   │   "Uncaught exception in ZMQStream ca │                 
                             │                                                                   │                 
                             │ /Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/pyth │                 
                             │ on3.12/site-packages/ipykernel/kernelbase.py:348 in               │                 
                             │ dispatch_control                                                  │                 
                             │                                                                   │                 
                             │    345 │   │   """Dispatch a control request, ensuring only one m │                 
                             │        time."""                                                   │                 
                             │    346 │   │   # Ensure only one control message is processed at  │                 
                             │    347 │   │   async with self._control_lock:                     │                 
                             │ ❱  348 │   │   │   await self.process_control(msg)                │                 
                             │    349 │                                                          │                 
                             │    350 │   async def process_control(self, msg):                  │                 
                             │    351 │   │   """dispatch control requests"""                    │                 
                             │                                                                   │                 
                             │ /Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/pyth │                 
                             │ on3.12/site-packages/ipykernel/kernelbase.py:354 in               │                 
                             │ process_control                                                   │                 
                             │                                                                   │                 
                             │    351 │   │   """dispatch control requests"""                    │                 
                             │    352 │   │   if not self.session:                               │                 


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 348, in dispatch_control
    await self.process_control(msg)
  File "/Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 354, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list


                    ERROR    Uncaught exception in ZMQStream callback                              ]8;id=510784;file:///Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py\zmqstream.py]8;;\:]8;id=452409;file:///Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py#567\567]8;;\
                             ╭──────────────── Traceback (most recent call last) ────────────────╮                 
                             │ /Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/pyth │                 
                             │ on3.12/site-packages/zmq/eventloop/zmqstream.py:565 in _log_error │                 
                             │                                                                   │                 
                             │   562 │   │   │   # handle async callbacks                        │                 
                             │   563 │   │   │   def _log_error(f):                              │                 
                             │   564 │   │   │   │   try:                                        │                 
                             │ ❱ 565 │   │   │   │   │   f.result()                              │                 
                             │   566 │   │   │   │   except Exception:                           │                 
                             │   567 │   │   │   │   │   gen_log.error(                          │                 
                             │   568 │   │   │   │   │   │   "Uncaught exception in ZMQStream ca │                 
                             │                                                                   │                 
                             │ /Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/pyth │                 
                             │ on3.12/site-packages/ipykernel/kernelbase.py:348 in               │                 
                             │ dispatch_control                                                  │                 
                             │                                                                   │                 
                             │    345 │   │   """Dispatch a control request, ensuring only one m │                 
                             │        time."""                                                   │                 
                             │    346 │   │   # Ensure only one control message is processed at  │                 
                             │    347 │   │   async with self._control_lock:                     │                 
                             │ ❱  348 │   │   │   await self.process_control(msg)                │                 
                             │    349 │                                                          │                 
                             │    350 │   async def process_control(self, msg):                  │                 
                             │    351 │   │   """dispatch control requests"""                    │                 
                             │                                                                   │                 
                             │ /Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/pyth │                 
                             │ on3.12/site-packages/ipykernel/kernelbase.py:354 in               │                 
                             │ process_control                                                   │                 
                             │                                                                   │                 
                             │    351 │   │   """dispatch control requests"""                    │                 
                             │    352 │   │   if not self.session:                               │                 


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 348, in dispatch_control
    await self.process_control(msg)
  File "/Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 354, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list


                    ERROR    Uncaught exception in ZMQStream callback                              ]8;id=373279;file:///Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py\zmqstream.py]8;;\:]8;id=718659;file:///Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py#567\567]8;;\
                             ╭──────────────── Traceback (most recent call last) ────────────────╮                 
                             │ /Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/pyth │                 
                             │ on3.12/site-packages/zmq/eventloop/zmqstream.py:565 in _log_error │                 
                             │                                                                   │                 
                             │   562 │   │   │   # handle async callbacks                        │                 
                             │   563 │   │   │   def _log_error(f):                              │                 
                             │   564 │   │   │   │   try:                                        │                 
                             │ ❱ 565 │   │   │   │   │   f.result()                              │                 
                             │   566 │   │   │   │   except Exception:                           │                 
                             │   567 │   │   │   │   │   gen_log.error(                          │                 
                             │   568 │   │   │   │   │   │   "Uncaught exception in ZMQStream ca │                 
                             │                                                                   │                 
                             │ /Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/pyth │                 
                             │ on3.12/site-packages/ipykernel/kernelbase.py:348 in               │                 
                             │ dispatch_control                                                  │                 
                             │                                                                   │                 
                             │    345 │   │   """Dispatch a control request, ensuring only one m │                 
                             │        time."""                                                   │                 
                             │    346 │   │   # Ensure only one control message is processed at  │                 
                             │    347 │   │   async with self._control_lock:                     │                 
                             │ ❱  348 │   │   │   await self.process_control(msg)                │                 
                             │    349 │                                                          │                 
                             │    350 │   async def process_control(self, msg):                  │                 
                             │    351 │   │   """dispatch control requests"""                    │                 
                             │                                                                   │                 
                             │ /Users/hritik.raj/fifth_elephant_workshop/module-02/venv/lib/pyth │                 
                             │ on3.12/site-packages/ipykernel/kernelbase.py:354 in               │                 
                             │ process_control                                                   │                 
                             │                                                                   │                 
                             │    351 │   │   """dispatch control requests"""                    │                 
                             │    352 │   │   if not self.session:                               │                 


---

## Step 6: Run the Server

The file `server.py` contains the exact same three tools packaged as a runnable MCP server using **Streamable HTTP** transport.

```bash
python server.py
```

This starts the server on `http://localhost:8000` with:
- **MCP endpoint:** `http://localhost:8000/mcp`
- **Health check:** `http://localhost:8000/health`

## Step 7: Connect to Cursor

Start the server (`python server.py`), then add this to your `.cursor/mcp.json`:

```json
{
  "mcpServers": {
    "github-tools": {
      "url": "http://localhost:8000/mcp"
    }
  }
}
```

Once registered, you can ask Cursor things like:
- *"What open pull requests are there across all my repositories?"*
- *"List the repos for octocat"*
- *"Review PR #42 on my-org/my-repo and leave a comment"*

---

## Summary

In this module you learned:

1. **`FastMCP`** — create a server with a single line: `mcp = FastMCP("name")`
2. **`@mcp.tool()`** — register tools by decorating async functions; the SDK generates the JSON Schema from type hints + docstrings
3. **`list_tools`** — how clients discover what the server can do
4. **`call_tool`** — how clients invoke a tool and get results back (returns `content_blocks, output`)
5. **Running & connecting** — `mcp.http_app()` + uvicorn serves Streamable HTTP; connect from Cursor with just a URL

---

## Next Steps

Move on to **Module 3** to connect an MCP client to this server and build a full agent loop!